In [ ]:

# import pandas as pd
# import numpy as np
# import os

# # Load pressure analysis dataset
# df = pd.read_csv("processed/pressure_analysis.csv")

# # Check required columns
# required = [
#     "striker_id", "striker_name", "match_id",
#     "runs", "is_wicket", "dismissed_player_id",
#     "extras_type", "pressure"
# ]

# for col in required:
#     if col not in df.columns:
#         raise ValueError(f"Missing column: {col}")

# # Clean data
# df["runs"] = pd.to_numeric(df["runs"], errors="coerce").fillna(0)

# df["extras_type"] = (
#     df["extras_type"].fillna("").astype(str).str.lower()
# )

# df["is_wicket"] = (
#     df["is_wicket"].astype(str).str.lower()
#     .isin(["true", "1", "yes"])
# )

# df["pressure"] = (
#     df["pressure"].astype(str).str.lower()
#     .isin(["true", "1", "yes"])
# )

# # Batter does not face a ball for a wide
# df["batter_ball"] = (
#     ~df["extras_type"].str.contains("wide", na=False)
# ).astype(int)

# # Count dismissal only if the striker was dismissed
# df["striker_out"] = (
#     df["is_wicket"]
#     & df["dismissed_player_id"].notna()
#     & (
#         pd.to_numeric(df["dismissed_player_id"], errors="coerce")
#         == pd.to_numeric(df["striker_id"], errors="coerce")
#     )
# ).astype(int)

# # Calculate overall performance
# overall = df.groupby(
#     ["striker_id", "striker_name"]
# ).agg(
#     overall_runs=("runs", "sum"),
#     overall_balls=("batter_ball", "sum"),
#     overall_outs=("striker_out", "sum")
# ).reset_index()

# overall["overall_sr"] = (
#     100 * overall["overall_runs"] /
#     overall["overall_balls"].replace(0, np.nan)
# )

# overall["overall_average"] = (
#     overall["overall_runs"] /
#     overall["overall_outs"].replace(0, np.nan)
# )

# # Calculate pressure performance
# p = df[df["pressure"]].copy()

# pressure = p.groupby(
#     ["striker_id", "striker_name"]
# ).agg(
#     pressure_runs=("runs", "sum"),
#     pressure_balls=("batter_ball", "sum"),
#     pressure_matches=("match_id", "nunique"),
#     pressure_outs=("striker_out", "sum")
# ).reset_index()

# pressure["pressure_sr"] = (
#     100 * pressure["pressure_runs"] /
#     pressure["pressure_balls"].replace(0, np.nan)
# )

# pressure["pressure_average"] = (
#     pressure["pressure_runs"] /
#     pressure["pressure_outs"].replace(0, np.nan)
# )

# # Compare pressure performance with overall performance
# result = pressure.merge(
#     overall,
#     on=["striker_id", "striker_name"],
#     how="left"
# )

# result["sr_uplift"] = (
#     result["pressure_sr"] - result["overall_sr"]
# )

# result["average_uplift"] = (
#     result["pressure_average"] - result["overall_average"]
# )

# # Minimum eligibility requirements
# result = result[
#     (result["pressure_balls"] >= 30) &
#     (result["pressure_matches"] >= 5)
# ].copy()

# # Calculate standard scores
# def zscore(series):
#     sd = series.std(ddof=0)

#     if pd.isna(sd) or sd == 0:
#         return pd.Series(0.0, index=series.index)

#     return (series - series.mean()) / sd

# result["sr_z"] = zscore(result["pressure_sr"])
# result["avg_z"] = zscore(result["pressure_average"])
# result["sr_uplift_z"] = zscore(result["sr_uplift"])
# result["avg_uplift_z"] = zscore(result["average_uplift"])

# # Weighted batter score
# raw_score = (
#     0.45 * result["sr_z"] +
#     0.25 * result["avg_z"] +
#     0.20 * result["sr_uplift_z"] +
#     0.10 * result["avg_uplift_z"]
# )

# # Confidence adjustment
# confidence = (
#     result["pressure_balls"] /
#     (result["pressure_balls"] + 40)
# ) * (
#     result["pressure_matches"] /
#     (result["pressure_matches"] + 3)
# )

# result["rating"] = raw_score * confidence

# # Rank batters
# result = result.sort_values(
#     ["rating", "pressure_balls"],
#     ascending=[False, False]
# ).reset_index(drop=True)

# result.insert(0, "rank", range(1, len(result) + 1))

# # Save results
# os.makedirs("output", exist_ok=True)

# result.head(10).to_csv(
#     "output/top_10_batters.csv",
#     index=False
# )

# print("\nTOP 10 BATTERS")
# print(
#     result[
#         ["rank", "striker_name", "pressure_runs",
#          "pressure_balls", "pressure_sr",
#          "pressure_average", "rating"]
#     ].head(10).to_string(index=False)
# )

# print("\nBatter ranking completed!")
# print("Saved: output/top_10_batters.csv")